# Denoising of '/home/jupyter-vruiz/gdrive_TomogramDenoising/tomograms/Corbel2301_block2_June2019_crop_june2024_ali_crop.mrc' ("corbel_big") using CryoCARE "Even/Odd"

* Inputs: `even.mrc` and `odd.mrc`, the pair of noisy vols, generated from the original one.
* Outputs: `denoised/Corbel2301_block2_June2019_crop_june2024_ali_crop.mrc` with the denoised vol, and `denoised.pdf` with a view of a tile of a central slice in Z.

In [ ]:
from pathlib import Path
from collections import namedtuple
import json

In [ ]:
Args = namedtuple("args", ["original", "even", "odd", "denoised"])
args = Args("/home/jupyter-vruiz/gdrive_TomogramDenoising/tomograms/Corbel2301_block2_June2019_crop_june2024_ali_crop.mrc",
            "even.mrc", "odd.mrc", "denoised/Corbel2301_block2_June2019_crop_june2024_ali_crop.mrc")

In [ ]:
if Path(args.denoised).exists():
    raise Exception(f"{args.denoised} already exists ... exiting")

In [ ]:
if not Path(args.even).exists() or not Path(args.odd):
    %run split_even_odd.ipynb

## Configure cryoCARE

In [ ]:
_ = {
    "even": [args.even],
    "odd": [args.odd],
    "mask": [""],
    "patch_shape": [32, 32, 32],
    "num_slices": 800,
    "split": 0.9,
    "tilt_axis": "Y",
    "n_normalization_samples": 200,
    "path": "./data",
    "overwrite": "True"
}

with open("train_data_config.json", 'w') as f:
    json.dump(_, f, indent=4)

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/empiar10311
source ~/envs/cryoCARE/bin/activate
cryoCARE_extract_train_data.py --conf train_data_config.json

## Train

In [ ]:
%%writefile train_config__evenodd.json
{
  "train_data": "./data",
  "epochs": 50,
  "steps_per_epoch": 200,
  "batch_size": 16,
  "unet_kern_size": 3,
  "unet_n_depth": 3,
  "unet_n_first": 16,
  "learning_rate": 0.0004,
  "model_name": "model",
  "path": "./",
  "gpu_id": [1]
}

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/corbel_big
#pwd
source ~/envs/cryoCARE/bin/activate
#cryoCARE_extract_train_data.py --conf train_data_config__evenodd.json
cryoCARE_train.py --conf train_config__evenodd.json

## Infer

In [ ]:
_ = {
    "path": "./model.tar.gz",
    "even": [args.original], 
    "odd": [args.original],
    "n_tiles": [2,2,2],
    "output": "denoised",
    "overwrite": "True",
    "gpu_id": [1]
}

with open("predict_config.json", 'w') as f:
    json.dump(_, f, indent=4)

%%writefile predict_config__evenodd.json
{
    "path": "./model.tar.gz",
    "even": ["corbel_big.mrc"], 
    "odd": ["corbel_big.mrc"],
    "n_tiles": [2,2,2],
    "output": "even_odd_denoised",
    "overwrite": "True",
    "gpu_id": [1]
}

In [ ]:
%%bash
#cd /nas/vruiz/cryoCARE/corbel_big
pwd
source ~/envs/cryoCARE/bin/activate
cryoCARE_predict.py --conf predict_config.json || true

In [ ]:
import mrcfile
import numpy as np
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt

In [ ]:
def read_MRC(file_path):
    return mrcfile.read(file_path)

In [ ]:
denoised_volume = read_MRC(args.denoised)

In [ ]:
#mrc_file_path = '/nas/vruiz/cryoCARE/corbel_big/corbel_big.mrc'
original_volume = read_MRC(args.original)

In [ ]:
original_volume.shape

In [ ]:
#mrc_file_path = 'even_odd_denoised/corbel_big.mrc'
denoised_volume = read_MRC(args.denoised)

In [ ]:
denoised_volume.shape

In [ ]:
# Choose a slice index in the middle of the volume for a good comparison
slice_idx = original_volume.shape[0] // 2

fig, axes = plt.subplots(1, 2, figsize=(20, 20))

# Plot the original slice z
im1 = axes[0].imshow(original_volume[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[0].set_title(f'Original Slice Z={slice_idx}')
axes[0].grid(False)

# Plot the original slice z+1
im2 = axes[1].imshow(denoised_volume[slice_idx, :, :].T, cmap='gray', origin='lower')
axes[1].set_title(f'N2N Even/Odd Denoised Slice Z={slice_idx}')
axes[1].grid(False)

plt.tight_layout()
plt.show()

In [ ]:
from matplotlib.pyplot import figure
figure(figsize=(16, 16))
slice_idx = denoised_volume.shape[0]//2
plt.imshow(denoised_volume[slice_idx, 0:400, 400:800], cmap="gray")
plt.savefig("denoised.pdf", bbox_inches='tight')